# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sachingowda8431/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
from google.colab import userdata
from datasets import load_dataset, get_dataset_config_names

HF_TOKEN = userdata.get("HF_TOKEN")

configs = get_dataset_config_names(
    "FlyRank/internship-warehouse",
    token=HF_TOKEN
)

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    configs[0],
    token=HF_TOKEN
)

split_name = list(dataset.keys())[0]

df = dataset[split_name].to_pandas()

print(df.shape)
df.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

(104, 9)


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Display dataset information
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nAvailable Columns:")
print(df.columns.tolist())

Rows: 104
Columns: 9

Available Columns:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Example baseline score
df["baseline_score"] = 0

if "impressions_90d" in df.columns:
    df["baseline_score"] += df["impressions_90d"]

if "days_since_last_update" in df.columns:
    df["baseline_score"] += df["days_since_last_update"]

df["reason_code"] = "RC001"
df["action_label"] = "Review for Content Refresh"

queue = df.sort_values("baseline_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
queue.head(10)

CSV saved successfully.


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start,baseline_score,reason_code,action_label
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22,0,RC001,Review for Content Refresh
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None,0,RC001,Review for Content Refresh
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06,0,RC001,Review for Content Refresh
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None,0,RC001,Review for Content Refresh
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None,0,RC001,Review for Content Refresh
5,client_08d2847f24cf89c1,True,True,True,gsc_and_ga4,2025-07-11,2026-06-28,2025-07-21,2026-02-19,0,RC001,Review for Content Refresh
6,client_0b245132bb722950,False,True,True,gsc_and_ga4,2026-02-25,2026-06-29,2026-04-12,2026-04-24,0,RC001,Review for Content Refresh
7,client_0e1acc6cd57b0eba,True,True,True,gsc_and_ga4,2025-06-25,2026-06-27,2025-09-24,2026-02-19,0,RC001,Review for Content Refresh
8,client_0fa64a184f18a4a0,False,True,True,gsc_and_ga4,2026-01-26,2026-06-29,2026-02-19,2026-02-17,0,RC001,Review for Content Refresh
9,client_123b42d7ca0e1690,True,False,False,no_search_or_analytics_access,2026-06-29,2026-07-02,None,None,0,RC001,Review for Content Refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)

review = pd.DataFrame({
    "Action": ["Review for Content Refresh"] * len(top20),
    "Reason Code": ["RC001"] * len(top20),
    "Confidence": ["Medium"] * len(top20),
    "What would make it wrong":
        ["Recent update or temporary traffic fluctuation"] * len(top20)
})

print(review)

                        Action Reason Code Confidence  \
0   Review for Content Refresh       RC001     Medium   
1   Review for Content Refresh       RC001     Medium   
2   Review for Content Refresh       RC001     Medium   
3   Review for Content Refresh       RC001     Medium   
4   Review for Content Refresh       RC001     Medium   
5   Review for Content Refresh       RC001     Medium   
6   Review for Content Refresh       RC001     Medium   
7   Review for Content Refresh       RC001     Medium   
8   Review for Content Refresh       RC001     Medium   
9   Review for Content Refresh       RC001     Medium   
10  Review for Content Refresh       RC001     Medium   
11  Review for Content Refresh       RC001     Medium   
12  Review for Content Refresh       RC001     Medium   
13  Review for Content Refresh       RC001     Medium   
14  Review for Content Refresh       RC001     Medium   
15  Review for Content Refresh       RC001     Medium   
16  Review for Content Refresh 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")

excluded = [
    "trend_direction",
    "trend_pct",
    "label"
]

for col in excluded:
    if col in df.columns:
        print(col, "exists but was NOT used.")
    else:
        print(col, "not present.")

print("\nBaseline completed successfully.")

Leakage Check
trend_direction not present.
trend_pct not present.
label not present.

Baseline completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.